In [13]:
# Self-supervised pretraining using SimCLR (via lightly)

import torch
from lightly.data import LightlyDataset
from lightly.transforms import SimCLRTransform
from lightly.models import SimCLR
from torchvision.models import resnet18
from lightly.loss import NTXentLoss
from lightly.data.collate import BaseCollateFunction
from torch.utils.data import DataLoader
from torchvision import transforms
import os
from PIL import Image

# === Config ===
data_dir = 'some_images'  # Folder with unlabeled images
batch_size = 3
epochs = 2
learning_rate = 1e-3
checkpoint_path = 'pretrained_streamflow_ai.pth'

# === Transforms ===
# bottom crop: removes 150 pixels from the bottom
def crop_bottom(img: Image.Image, pixels: int = 150):
    return img.crop((0, 0, img.width, img.height - pixels))

# create SimCLR-style view with bottom-cropping first
class BottomCropSimCLRTransform:
    def __init__(self, crop_pixels=150, input_size=224):
        self.crop_pixels = crop_pixels
        self.simclr_transform = SimCLRTransform(input_size=input_size)

    def __call__(self, img):  # Expect just a single image, not a list
        if not isinstance(img, Image.Image):
            raise TypeError(f"Expected PIL Image, got {type(img)}")
        cropped = crop_bottom(img, self.crop_pixels)
        return self.simclr_transform(cropped)
        
# Instantiate the full transform
transform = BottomCropSimCLRTransform(crop_pixels=150, input_size=224)

# === Dataset & Dataloader ===
dataset = LightlyDataset(input_dir='some_images')
collate_fn = BaseCollateFunction(transform=transform)
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

# === Model, Loss, Optimizer ===
backbone = resnet18(pretrained=False)
model = SimCLR(backbone, num_ftrs=backbone.fc.in_features)
loss_fn = NTXentLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# === Training loop ===
for epoch in range(epochs):
    total_loss = 0
    model.train()
    for (x0, x1), _, _ in dataloader:
        x0, x1 = x0.to(device), x1.to(device)
        z0, z1 = model(x0), model(x1)
        loss = loss_fn(z0, z1)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {total_loss/len(dataloader):.4f}")

# === Save encoder weights ===
torch.save(model.backbone.state_dict(), checkpoint_path)
print(f"Saved pretrained encoder to {checkpoint_path}")


AttributeError: 'list' object has no attribute 'unsqueeze_'